In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# วัด Qubit

<details>
<summary><b>เวอร์ชันของ package</b></summary>

โค้ดในหน้านี้พัฒนาโดยใช้ requirement ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
เพื่อรับข้อมูลเกี่ยวกับสถานะของ Qubit สามารถ _วัด_ มันลงใน [classical bit](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Clbit) ได้ ใน Qiskit การวัดทำในฐาน computational basis นั่นคือฐาน Pauli-$Z$ แบบ single-qubit ดังนั้น การวัดจะให้ผลลัพธ์ 0 หรือ 1 ขึ้นอยู่กับการซ้อนทับกับ eigenstate ของ Pauli-$Z$ คือ $|0\rangle$ และ $|1\rangle$:

$$
|q\rangle \xrightarrow{measure}\begin{cases}
      0 (\text{outcome}+1), \text{with probability } p_0=|\langle q|0\rangle|^{2}\text{,} \\
      1 (\text{outcome}-1), \text{with probability } p_1=|\langle q|1\rangle|^{2}\text{.}
    \end{cases}
$$

## การวัดกลางวงจร
การวัดกลางวงจร (mid-circuit measurements) เป็นองค์ประกอบสำคัญของ dynamic circuit ก่อนหน้า `qiskit-ibm-runtime` v0.43.0 `measure` เป็นคำสั่งการวัดเพียงอย่างเดียวใน Qiskit อย่างไรก็ตาม การวัดกลางวงจรมีความต้องการการปรับแต่งที่แตกต่างจากการวัด _terminal_ (การวัดที่เกิดขึ้นที่ท้าย Circuit) ตัวอย่างเช่น ต้องพิจารณาระยะเวลาของคำสั่งเมื่อปรับแต่งการวัดกลางวงจร เพราะคำสั่งที่ยาวกว่าทำให้ Circuit มีสัญญาณรบกวนมากขึ้น ไม่จำเป็นต้องพิจารณาระยะเวลาคำสั่งสำหรับการวัด terminal เพราะไม่มีคำสั่งหลังจากการวัด terminal

ใน `qiskit-ibm-runtime` v0.43.0 คำสั่ง `MidCircuitMeasure` ได้รับการแนะนำ ตามชื่อที่บ่งบอก มันเป็นคำสั่งการวัดใหม่ที่ได้รับการปรับให้เหมาะสมสำหรับการวัดกลางวงจรบน IBM&reg; QPU

> **Note:** คำสั่ง `MidCircuitMeasure` map ไปยังคำสั่ง `measure_2` ที่รายงานใน `supported_instructions` ของ Backend อย่างไรก็ตาม `measure_2` ไม่รองรับบน Backend ทั้งหมด ใช้ `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` เพื่อค้นหา Backend ที่รองรับ อาจมีการเพิ่มการวัดใหม่ในอนาคต แต่ไม่มีการรับประกัน

## ใช้การวัดกับ Circuit
มีหลายวิธีในการใช้การวัดกับ Circuit:

### วิธี `QuantumCircuit.measure`
ใช้วิธี [`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure) เพื่อวัด [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class)

ตัวอย่าง:

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(5, 5)
qc.x(0)
qc.x(1)
qc.x(4)
qc.measure(
    range(5), range(5)
)  #  Measures all qubits into the corresponding clbit.

In [2]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure(1, 0)  # Measure qubit 1 into the classical bit 0.

### `Measure` class

The Qiskit [Measure](/docs/api/qiskit/circuit#qiskit.circuit.Measure) class measures the specified qubits.

In [3]:
from qiskit.circuit import Measure

qc = QuantumCircuit(3, 1)
qc.x([0, 1])
qc.append(Measure(), [0], [0])  # measure qubit 0 into clbit 0

### `QuantumCircuit.measure_all` method

To measure all qubits into the corresponding classical bits, use the [`measure_all`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) method. By default, this method adds new classical bits in a `ClassicalRegister` to store these measurements.

In [4]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_all()  # Measure all qubits.

### คลาส `Measure`
คลาส [Measure](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Measure) ของ Qiskit วัด Qubit ที่ระบุ

In [5]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_active()  # Measure qubits that are not idle, that is, qubits 0 and 2.

<span id="midcircuit"></span>
### `MidCircuitMeasure` method


Use `MidCircuitMeasure` to apply a mid-circuit measurement (requires `qiskit-ibm-runtime` v0.43.0 or later).  While you can use `QuantumCircuit.measure` for a mid-circuit measurement, because of its design, `MidCircuitMeasure` is typically a better choice.  For example, it adds less overhead to your circuit than when using `QuantumCircuit.measure`.

In [6]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.circuit import MidCircuitMeasure
from qiskit.circuit import Measure

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circ = QuantumCircuit(2, 2)
circ.x(0)
circ.append(MidCircuitMeasure(), [0], [0])
# circ.measure([0], [0])
# circ.measure_all()
print(circ.draw(cregbundle=False))

     ┌───┐┌────────────┐
q_0: ┤ X ├┤0           ├
     └───┘│            │
q_1: ─────┤  Measure_2 ├
          │            │
c_0: ═════╡0           ╞
          └────────────┘
c_1: ═══════════════════
                        


### วิธี `QuantumCircuit.measure_all`
เพื่อวัด Qubit ทั้งหมดลงใน classical bit ที่สอดคล้องกัน ใช้วิธี [`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) โดยค่าเริ่มต้น วิธีนี้จะเพิ่ม classical bit ใหม่ใน `ClassicalRegister` เพื่อเก็บการวัดเหล่านี้